# 01 - Recolección de Datos

**Objetivo**: Descargar datos de tasas de fertilidad y variables relacionadas desde fuentes públicas.

**Fuentes principales**:
- Banco Mundial (World Development Indicators)
- Naciones Unidas (si es necesario)

---

## 1. Setup - Importar Bibliotecas

In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# World Bank API
import wbgapi as wb

# Utilities
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ Bibliotecas importadas correctamente")
print(f"Pandas version: {pd.__version__}")
print(f"Fecha de ejecución: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Bibliotecas importadas correctamente
Pandas version: 3.0.0
Fecha de ejecución: 2026-02-06 22:09:08


## 2. Verificar Conexión con la API del Banco Mundial

In [2]:
# Test: Buscar indicadores relacionados con fertilidad
# Esto nos ayuda a encontrar los códigos correctos

# Verificar conexión con la API del Banco Mundial
print("🔄 Verificando conexión con el Banco Mundial...")

try:
    # Probar conexión listando algunos indicadores
    fertility_info = wb.series.get('SP.DYN.TFRT.IN')
    print(f"✅ Conexión exitosa!")
    print(f"\n📊 Indicador de prueba:")
    print(f"   Código: SP.DYN.TFRT.IN")
    print(f"   Nombre: {fertility_info['value']}")
    print("\n✅ API del Banco Mundial funcionando correctamente")
except Exception as e:
    print(f"❌ Error de conexión: {e}")

🔄 Verificando conexión con el Banco Mundial...
✅ Conexión exitosa!

📊 Indicador de prueba:
   Código: SP.DYN.TFRT.IN
   Nombre: Fertility rate, total (births per woman)

✅ API del Banco Mundial funcionando correctamente


## 3. Definir Indicadores a Descargar

Basados en nuestro documento `data_sources.md`

In [3]:
# Diccionario de indicadores del Banco Mundial
indicators = {
    'SP.DYN.TFRT.IN': 'fertility_rate',           # Tasa de fertilidad total
    'NY.GDP.PCAP.CD': 'gdp_per_capita',           # PIB per cápita
    'SL.UEM.TOTL.ZS': 'unemployment_rate',        # Tasa de desempleo
    'SE.SEC.ENRR.FE': 'female_education',         # Matrícula secundaria femenina
    'SL.TLF.CACT.FE.ZS': 'female_labor_force',    # Participación laboral femenina
    'SP.URB.TOTL.IN.ZS': 'urbanization_rate',     # Tasa de urbanización
}

# Período de análisis
START_YEAR = 2000
END_YEAR = 2023

print(f"📊 Indicadores a descargar: {len(indicators)}")
for code, name in indicators.items():
    print(f"  - {name}: {code}")
print(f"\n📅 Período: {START_YEAR} - {END_YEAR}")

📊 Indicadores a descargar: 6
  - fertility_rate: SP.DYN.TFRT.IN
  - gdp_per_capita: NY.GDP.PCAP.CD
  - unemployment_rate: SL.UEM.TOTL.ZS
  - female_education: SE.SEC.ENRR.FE
  - female_labor_force: SL.TLF.CACT.FE.ZS
  - urbanization_rate: SP.URB.TOTL.IN.ZS

📅 Período: 2000 - 2023


## 4. Descargar Datos

**Nota**: Este proceso puede tomar varios minutos dependiendo de tu conexión.

In [4]:
# Descargar datos del Banco Mundial - Versión mejorada
print("🔄 Iniciando descarga de datos del Banco Mundial...")
print("⏳ Descargando con método alternativo más estable...\n")

import time

# Diccionario para almacenar los dataframes
datasets = {}

# Descargar cada indicador con manejo de errores mejorado
for code, name in indicators.items():
    try:
        print(f"📥 Descargando: {name}...")
        
        # Método alternativo: descargar sin labels primero
        data = wb.data.DataFrame(
            code, 
            time=range(START_YEAR, END_YEAR + 1),
            labels=False,
            numericTimeKeys=True
        )
        
        # Guardar en el diccionario
        datasets[name] = data
        
        print(f"   ✅ Descargado: {data.shape[0]} países × {data.shape[1]} años")
        
        # Pequeña pausa para no saturar la API
        time.sleep(0.5)
        
    except Exception as e:
        print(f"   ❌ Error: {str(e)[:100]}")
        continue

print(f"\n🎉 ¡Descarga completada!")
print(f"📊 Datasets descargados exitosamente: {len(datasets)}/{len(indicators)}")

🔄 Iniciando descarga de datos del Banco Mundial...
⏳ Descargando con método alternativo más estable...

📥 Descargando: fertility_rate...
   ✅ Descargado: 266 países × 24 años
📥 Descargando: gdp_per_capita...
   ✅ Descargado: 266 países × 24 años
📥 Descargando: unemployment_rate...
   ✅ Descargado: 266 países × 24 años
📥 Descargando: female_education...
   ✅ Descargado: 266 países × 24 años
📥 Descargando: female_labor_force...
   ✅ Descargado: 266 países × 24 años
📥 Descargando: urbanization_rate...
   ✅ Descargado: 266 países × 24 años

🎉 ¡Descarga completada!
📊 Datasets descargados exitosamente: 6/6


## 5. Guardar Datos Raw

Los datos descargados se guardarán en `data/raw/` sin modificaciones.

In [5]:
# Guardar los datos descargados en archivos CSV
import os
from datetime import datetime

# Crear carpeta data/raw si no existe
os.makedirs('../data/raw', exist_ok=True)

print("💾 Guardando datos en data/raw/...")

# Fecha para el nombre de archivo
fecha = datetime.now().strftime('%Y%m%d')

# Guardar cada dataset
for name, data in datasets.items():
    filename = f'../data/raw/worldbank_{name}_{fecha}.csv'
    data.to_csv(filename)
    print(f"   ✅ Guardado: {filename}")
    print(f"      Dimensiones: {data.shape[0]} filas × {data.shape[1]} columnas")

print(f"\n🎉 ¡Todos los datos guardados exitosamente!")
print(f"📁 Ubicación: data/raw/")
print(f"📊 Total de archivos: {len(datasets)}")

💾 Guardando datos en data/raw/...
   ✅ Guardado: ../data/raw/worldbank_fertility_rate_20260206.csv
      Dimensiones: 266 filas × 24 columnas
   ✅ Guardado: ../data/raw/worldbank_gdp_per_capita_20260206.csv
      Dimensiones: 266 filas × 24 columnas
   ✅ Guardado: ../data/raw/worldbank_unemployment_rate_20260206.csv
      Dimensiones: 266 filas × 24 columnas
   ✅ Guardado: ../data/raw/worldbank_female_education_20260206.csv
      Dimensiones: 266 filas × 24 columnas
   ✅ Guardado: ../data/raw/worldbank_female_labor_force_20260206.csv
      Dimensiones: 266 filas × 24 columnas
   ✅ Guardado: ../data/raw/worldbank_urbanization_rate_20260206.csv
      Dimensiones: 266 filas × 24 columnas

🎉 ¡Todos los datos guardados exitosamente!
📁 Ubicación: data/raw/
📊 Total de archivos: 6


## 6. Resumen de Datos Descargados

In [6]:
# Resumen de los datos descargados
print("📊 RESUMEN DE DATOS DESCARGADOS\n")
print("="*60)

for name, data in datasets.items():
    print(f"\n📈 {name.upper()}")
    print(f"   Dimensiones: {data.shape[0]} países × {data.shape[1]} años")
    print(f"   Período: {data.columns[0]} - {data.columns[-1]}")
    print(f"   Valores faltantes: {data.isna().sum().sum()} ({data.isna().sum().sum() / data.size * 100:.1f}%)")
    
    # Mostrar algunos países como ejemplo
    print(f"   Primeros 3 países:")
    print(f"      {data.index[0]}, {data.index[1]}, {data.index[2]}")

print("\n" + "="*60)
print("✅ Descarga automática completada exitosamente!")
print("📁 Los datos están listos en: data/raw/")
print("\n🎯 Siguiente paso: Notebook 02 - Limpieza de datos")

📊 RESUMEN DE DATOS DESCARGADOS


📈 FERTILITY_RATE
   Dimensiones: 266 países × 24 años
   Período: 2000 - 2023
   Valores faltantes: 24 (0.4%)
   Primeros 3 países:
      ABW, AFE, AFG

📈 GDP_PER_CAPITA
   Dimensiones: 266 países × 24 años
   Período: 2000 - 2023
   Valores faltantes: 191 (3.0%)
   Primeros 3 países:
      ABW, AFE, AFG

📈 UNEMPLOYMENT_RATE
   Dimensiones: 266 países × 24 años
   Período: 2000 - 2023
   Valores faltantes: 748 (11.7%)
   Primeros 3 países:
      ABW, AFE, AFG

📈 FEMALE_EDUCATION
   Dimensiones: 266 países × 24 años
   Período: 2000 - 2023
   Valores faltantes: 1856 (29.1%)
   Primeros 3 países:
      ABW, AFE, AFG

📈 FEMALE_LABOR_FORCE
   Dimensiones: 266 países × 24 años
   Período: 2000 - 2023
   Valores faltantes: 748 (11.7%)
   Primeros 3 países:
      ABW, AFE, AFG

📈 URBANIZATION_RATE
   Dimensiones: 266 países × 24 años
   Período: 2000 - 2023
   Valores faltantes: 24 (0.4%)
   Primeros 3 países:
      ABW, AFE, AFG

✅ Descarga automática complet

---

## ✅ Checklist

- [ ] Conexión con API del Banco Mundial establecida
- [ ] Indicadores identificados y verificados
- [ ] Datos de fertilidad descargados
- [ ] Datos socioeconómicos descargados
- [ ] Archivos guardados en `data/raw/`
- [ ] Resumen de datos generado

**Próximo notebook**: `02_data_cleaning.ipynb` - Limpieza y procesamiento de datos